# Retrieval evaluation ground truth generation

## Ground truth - search retrieval 
Here we create some golden truth so that we can measure the performance of the search
- pgvector_search
- pg_full_text_search
- pg_full_text_search_soft_match
- rrf - reciprocal ranked fusion

In [2]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

openai_client = OpenAI()

In [3]:
# load primary dataset from postgres
from utils.utils import get_connection
from psycopg.rows import dict_row

knowledge_base_results = []
with get_connection() as connection:
    connection.row_factory = dict_row
    knowledge_base_results = connection.execute(
        """
        SELECT 
            id,
            report_year,
            section,
            text
        FROM 
        knowledge_base_chunks
        WHERE report_year = 2025
        """).fetchall()

In [4]:

# print(knowledge_base_results[:5])
# print(len(knowledge_base_results))
# instructions
data_gen_instructions = """
You write search-evaluation questions for ONE chunk of an ABCWUA water quality report.

Ask only questions this chunk can answer. If the chunk is a header, caption, or table fragment, ask fewer questions (1-3).
Do not ask about source, treatment, monitoring, awards, lead, or "where do I find more info" unless that topic is actually in the chunk.
Prefer specific facts in the chunk (contaminant names, numbers, URLs, section headings) over generic water-safety questions.
Write like a resident asking on the internet: complete sentences, not too formal, not too short, not too long.
Paraphrase. Do not copy long phrases from the chunk.
""".strip()

In [5]:
# test a single chunk to start
print(knowledge_base_results[0])

from pydantic import BaseModel

# the format generated questions should be in
class Questions(BaseModel):
    questions: list[str]

import json


doc = knowledge_base_results[0]
user_prompt = json.dumps(doc)

messages = [
    {"role": "system", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt},
]


{'id': 184, 'report_year': 2025, 'section': 'page_5_chunk_1', 'text': 'from its water distribution system. however, the utility offers free lead and copper testing for customers concerned about their home plumbing fixtures. to schedule a test, visit www. abcwua. org / your - drinking - water - lead - sample - collection - request / for more information about the water authority ’ s current lead survey, see page 3. # # # # # # results of 2025 customer - requested lead testing ( 65 samples ) | substance | minimum | maximum detected | 90th percentile | action level | | - - - | - - - | - - - | - -'}


In [6]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini", input=messages, text_format=Questions
)

In [7]:
response.output_parsed.questions

['Does the utility offer free lead and copper testing for customers who are worried about their home plumbing?',
 'How do I schedule a lead sample collection test online?',
 'What was the 90th percentile result in the 2025 customer-requested lead testing for the 65 samples shown here?']

In [8]:
def format_questions(questions, chunk):
    records = []
    for q in questions:
        records.append({"question": q, "document": chunk["id"]})
    return records

def generate_ground_truth(chunk):
    # Build messages per chunk. The old version reused the first test
    # cell's `messages` (chunk 184), so every document got the same prompt.
    chunk_messages = [
        {"role": "system", "content": data_gen_instructions},
        {"role": "user", "content": json.dumps(chunk)},
    ]
    response = openai_client.responses.parse(
        model="gpt-5.4-mini", input=chunk_messages, text_format=Questions
    )
    return format_questions(response.output_parsed.questions, chunk)

In [9]:
# generate ground truth for all chunks
from tqdm.auto import tqdm
ground_truth = []
for chunk in tqdm(knowledge_base_results):
    ground_truth.append(generate_ground_truth(chunk))

/Users/testuser/Developer/bernalillo-water-rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 40/40 [00:46<00:00,  1.16s/it]


In [10]:
# export to csv as one row per {question, document}
import pandas as pd
ground_truth_df = pd.DataFrame(
    [item for recs in ground_truth for item in recs]
)
ground_truth_df.to_csv("../data/processed/search_ground_truth.csv", index=False)
ground_truth_df.head()

,question,document
0,How can I request a free lead and copper test ...,184
1,How many customer-requested lead samples were ...,184
2,"What were the minimum, maximum, and 90th perce...",184
3,What were the reported lead levels in this tab...,185
4,Does the chunk say anything about where lead i...,185
